# Assignment 1: Information Retrieval

Mert Erol, merol, 20-915-245

## Overview
In this assignment you will implement a BM25 retrieval system from scratch.
Given a text query, your system must find the most relevant documents from the **Cranfield Collection** — 
a foundational corpus for evaluating information retrieval systems containing:
- **1,400** documents
- **225** queries
- **1,837** relevance judgments (qrels)

## Rules
- You may only use **built-in Python libraries** plus `numpy` and `pandas`
- Full-packaged retrieval solutions (e.g. `rank_bm25`) are **not permitted**
- Do **not** modify function signatures or the skeleton class
- Do **not** change cells marked **do not edit**

## Deadline
**March 19, 23:59 CET**

## How to Use This Notebook
1. Run imports and BM25 Skeleton class first — always
2. Implement each task in the cells marked **TODO: your code here** and answer questions marked **TODO**
3. If you restart the kernel, re-run all cells from top to bottom

## Load the data

In [1]:
!pip install ir_datasets


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import ir_datasets
dataset = ir_datasets.load("cranfield")

### Inspect the queries

In [3]:
for query in dataset.queries_iter():
    print(query) # namedtuple<query_id, text>
    break

GenericQuery(query_id='1', text='what similarity laws must be obeyed when constructing aeroelastic models\nof heated high speed aircraft .')


### Inspect the documents

In [4]:
for doc in dataset.docs_iter():
    print(doc) # namedtuple<doc_id, title, text, author, bib>
    break

CranfieldDoc(doc_id='1', title='experimental investigation of the aerodynamics of a\nwing in a slipstream .', text='experimental investigation of the aerodynamics of a\nwing in a slipstream .\n  an experimental study of a wing in a propeller slipstream was\nmade in order to determine the spanwise distribution of the lift\nincrease due to slipstream at different angles of attack of the wing\nand at different free stream to slipstream velocity ratios .  the\nresults were intended in part as an evaluation basis for different\ntheoretical treatments of this problem .\n  the comparative span loading curves, together with\nsupporting evidence, showed that a substantial part of the lift increment\nproduced by the slipstream was due to a /destalling/ or\nboundary-layer-control effect .  the integrated remaining lift\nincrement, after subtracting this destalling lift, was found to agree\nwell with a potential flow theory .\n  an empirical evaluation of the destalling effects was made for\nthe s

### Inspect the query-document relevance judgments

In [5]:
for qrel in dataset.qrels_iter():
    print(qrel) # namedtuple<query_id, doc_id, relevance, iteration>
    break

TrecQrel(query_id='1', doc_id='184', relevance=2, iteration='0')


## Setup

The cells below set up the imports and define the BM25 skeleton class. 
You do not need to modify anything here — simply run both cells before starting Task 1.

In [6]:
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Set, Optional

import numpy as np

# NLTK for tokenization
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords", quiet=True)

True

**Note:** The `BM25` class below contains all the methods you will implement throughout this assignment.
Each task will ask you to implement one or more of these methods and attach them to the class.

In [7]:
# BM25 Skeleton (run this first, do not edit)

class BM25:
    """
    BM25 class:
      - tokenizer: tokenize(text) -> List[str]
      - retriever: retrieve(query_text, k) -> ranked list
      - evaluator: evaluate(metric, k) and grid_search over k1, b

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        
        # --- BM25 Hyperparameters ---
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # --- Corpus Statistics (populated by build()) ---
        self.N: int = 0                        # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]: raise NotImplementedError
    def build(self, docs: Iterable) -> None: raise NotImplementedError
    def idf(self, term: str) -> float: raise NotImplementedError
    def score(self, query_tokens: List[str], doc_id: str) -> float: raise NotImplementedError
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]: raise NotImplementedError
    def evaluate(self, queries_iter: Iterable, qrels_iter: Iterable, k: int = 10, metrics: Optional[List[str]] = None) -> Dict[str, float]: raise NotImplementedError
    def grid_search(self, queries_iter_factory, qrels_iter_factory, k: int = 10) -> List[Dict]: raise NotImplementedError



## Task 1: Tokenizer 

Implement a tokenizer that converts a text into a list of keywords: tokenize(text) -> List[str].

In [8]:
def tokenize(self, text: str) -> List[str]:
    """
    Convert raw text into a list of processed tokens.

    Requirements:
      1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
      2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
      3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
      4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
      5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

    Args:
        text: raw input string
    Returns:
        List of processed token strings

    Hints:
        - Perform text normalization using Python string methods.
        - Use the `re` module to extract alphanumeric tokens.
        - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

    """
    tokenizer = nltk.RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    
    filtered_tokens = [self._stemmer.stem(token) for token in tokens if token not in stop_words]
    return filtered_tokens
    
# attach to BM25 class
BM25.tokenize = tokenize

In [9]:
# ===============================================
# NOTE: Personal Testing Only — NOT Part of Assignment
# ===============================================
#
# This cell contains some basic example test cases for the `tokenize()` method.
# It is meant to help you check your own implementation.
# 
# It does NOT count toward your grade and is safe to run multiple times.
# You can modify it to test additional examples if you want.


bm25_test = BM25(remove_stopwords=True)

# Test 1: Lowercasing + stemming
assert bm25_test.tokenize("Working worked works") == ["work", "work", "work"]

# Test 2: Alphanumeric extraction + punctuation removal
assert bm25_test.tokenize("Hello, world 2026!") == ["hello", "world", "2026"]

# Test 3: Stopword removal + stemming
assert bm25_test.tokenize("The document has a title") == ["document", "titl"]

# Test 4: Mixed case + symbols + stemming
assert bm25_test.tokenize("Machine-Learning @ UZH :)") == ["machin", "learn", "uzh"]

print("All tokenizer tests passed")

All tokenizer tests passed


## Task 2: BM25 Retriever

Implement the document retriever using BM25 that finds the best top-k documents for a given query.

### Task 2a: Build Index

Implement `build()` to precompute the corpus statistics needed for BM25 scoring.

Once complete, the following attributes will be populated and used by `score()` and `retrieve()` in the next steps.

In [10]:

from collections import Counter

def build(self, docs_iter: Iterable) -> None:
    """
    Iterate over all documents and precompute corpus statistics needed for BM25.

    Args:
        docs_iter: iterable of Cranfield documents, each a namedtuple with fields:
                   (doc_id, title, text, author, bib)
    
    Requirements: 
        After calling build(), the following must be populated:
            self.N        : int                      — total number of documents
            self.avgdl    : float                    — average document length
            self.df       : Dict[str, int]           — term -> number of docs containing term
            self.doc_len  : Dict[str, int]           — doc_id -> document length
            self.doc_tf   : Dict[str, Counter]       — doc_id -> term frequencies per document
            self.docs_store: Dict[str, Dict[str, str]] — doc_id:str -> {"title": ..., "text": ...} (raw title & text per document)

    Note:
        Per-document steps (repeat for each document):
            1. Extract title and text safely
            2. Concatenate title and text, then tokenize using self.tokenize()
            3. Compute term frequencies  → store in self.doc_tf
            4. Compute document length → store in self.doc_len
            5. Store raw title and text → store in self.docs_store

        Corpus-level steps (after all documents):
            6. Count how many documents each term appears in → store in self.df
            (each term is counted once per document, not once per occurrence)
            7. Compute self.avgdl and self.N

    """
    counter = 0
    len_docs = 0
    
    # Step 1-5:
    for doc in docs_iter:        
        counter += 1
        
        title = doc.title or ""
        text = doc.text or ""
        title_and_text = title + " " + text
        
        tokens = self.tokenize(title_and_text)
        len_docs += len(tokens)
        
        self.docs_store[doc.doc_id] = {"title": title, "text": text}
        self.doc_len[doc.doc_id] = len(tokens)
        self.doc_tf[doc.doc_id] = Counter(tokens)
        
    # step 6:
    for doc_id, tf_counter in self.doc_tf.items():
        for term in tf_counter:
            self.df[term] = self.df.get(term, 0) + 1
            
    # step 7:
    self.N = counter
    self.avgdl = len_docs / counter

# attach to BM25 class
BM25.build = build

### Task 2b: BM25 Scoring

Now that the corpus statistics can be precomputed, implement the two core scoring functions of BM25.

- `idf(term)` computes how rare a term is across the corpus
- `score(query_tokens, doc_id)` combines IDF with term frequency to compute a BM25 score for a query-document pair


In [11]:
def idf(self, term: str) -> float:
    """
    Compute Inverse Document Frequency with standard smoothing.

    Args:
        term: a single token str

    Returns:
        float — IDF score for the term

    Formula:
        log( (N - df + 0.5) / (df + 0.5) + 1 )

    """

    # implement IDF formula
    df = self.df.get(term, 0)
    
    return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

# attach to BM25 class
BM25.idf = idf


def score(self, query_tokens: List[str], doc_id: str) -> float:
    """
    Compute BM25 score for a query against a single document.

    Args:
        query_tokens: list of tokenized query terms
        doc_id: document to score against

    Returns:
        float — BM25 score for this query-document pair

    Formula (sum over query terms):
        score(q, d) = sum over t in q [ IDF(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl/avg_dl)) ]

    where:
        q    = query
        d    = document
        t    = a query term
        f    = term frequency of t in document
        dl   = length of the document
        k1   = self.k1
        b    = self.b

    Hints:
        - skip terms with frequency 0
        - return 0.0 for empty documents
        - sum over query terms (you can keep duplicates, but set() is common and stable for students)
    """
    
    if doc_id not in self.doc_len or self.doc_len[doc_id] == 0:
        return 0.0
    
    score = 0.0
    dl = self.doc_len[doc_id]
    tf = self.doc_tf[doc_id]
    
    for term in set(query_tokens):
        f = tf.get(term, 0)
        
        if f == 0:
            continue
        
        df = self.df.get(term, 0)
        idf = math.log(1 + (self.N - df + 0.5) / (df + 0.5))
        
        denom = f + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))
        
        score += idf * (f * (self.k1 + 1)) / denom
        
    return score
    
    


# attach to BM25 class
BM25.score = score

### Task 2c: Retrieve

Implement `retrieve()` to return the top-k most relevant documents for a given query, ranked by BM25 score.

In [12]:
def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
    """
    Return the top-k documents for a given query, ranked by BM25 score.
    Brute-force over all documents.
    
    Args:
        query_text: raw query string
        k:          number of documents to return

    Returns:
        List of (doc_id, score) tuples sorted by score descending

    Hints:
        - tokenize the query first using self.tokenize()
        - iterate over all documents in self.doc_tf.keys()
        - use self.score() to compute the BM25 score for each document
        - only keep documents with a non-zero score
    """

    tokenized = self.tokenize(query_text)
    docs = []
    
    for doc_id in self.doc_tf.keys():
        score = self.score(tokenized, doc_id)
        if score > 0:
            docs.append((doc_id, score))

    # Sort documents by score in descending order
    docs.sort(key=lambda x: x[1], reverse=True)
    
    return docs[:k]

# attach to BM25 class
BM25.retrieve = retrieve

### Check Task 2: Implementing a BM25 retriever

Run the cell below to check your implementation. You should see:
- Total number of documents and average document length after building the index
- A ranked list of 10 documents for a sample query with their BM25 scores and titles


In [13]:
# Build index
bm25 = BM25()
print("Building index...")
bm25.build(dataset.docs_iter())
print(f"Done. N={bm25.N} docs, avgdl={bm25.avgdl:.2f}")

print('-'*120)

# Single query demo
q = next(dataset.queries_iter())
top10 = bm25.retrieve(q.text, k=10)
print("Query ID:", q.query_id, "Query Text:", q.text)
print('-'*50)
print("rank", "doc_id",  "score",  "title")
print('-'*50)
for rank, (doc_id, score) in enumerate(top10, 1):
    title = bm25.docs_store[doc_id]["title"]
    print(f"{rank:02d} {doc_id}  {score:.4f}  {title[:80]}")

Building index...
Done. N=1400 docs, avgdl=102.94
------------------------------------------------------------------------------------------------------------------------
Query ID: 1 Query Text: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .
--------------------------------------------------
rank doc_id score title
--------------------------------------------------
01 51  21.9151  theory of aircraft structural models subjected to aerodynamic
heating and extern
02 486  21.2723  similarity laws for aerothermoelastic testing .
03 12  18.5659  some structural and aerelastic considerations of high
speed flight .
04 184  17.8901  scale models for thermo-aeroelastic research .
05 573  17.0003  viscous hypersonic similitude .
06 878  16.5020  experimental model techniques and equipment for flutter
investigations .
07 665  14.4273  on the theory of hypersonic gas flow with a power law
shock wave .
08 746  14.1964  aeroelastic problems in

## Task 3: BM25 Evaluator

### Task 3a: Implement evaluate function for "recall", "precision", "mrr", "ndcg"

In [14]:
def evaluate(
    self,
    queries_iter: Iterable,
    qrels_iter: Iterable,
    k: int = 10,
    metrics: Optional[List[str]] = None,
) -> Dict[str, float]:
    """
    Compute evaluation metrics averaged over all queries.

    Args:
        queries_iter : iterable of queries (each has .query_id and .text)
        qrels_iter   : iterable of relevance judgments (each has .query_id, .doc_id, .relevance)
        k            : number of top documents to evaluate
        metrics      : list of metrics to compute, e.g. ["recall", "precision", "mrr", "ndcg"]
                       if None, compute all four

    Returns:
        Dict mapping metric name -> average score across all queries

    Note:
        Relevance:
            - treat relevance <= 0 as non-relevant (0)
            - treat relevance  > 0 as relevant (1)

        Metrics:
            Recall@k    = (number of relevant docs in top-k) / (total relevant docs for query)

            Precision@k = (number of relevant docs in top-k) / k

            MRR@k       = 1 / (rank of first relevant doc in top-k)
                        0 if no relevant doc found

            nDCG@k      = DCG@k / IDCG@k
                        DCG@k  = sum over rank i: rel_i / log2(i + 2)    (i is 0-indexed)
                        IDCG@k = DCG of the ideal ranking (sort all relevant docs by relevance desc)

    Hints:
        - define two helper functions inside evaluate():
            * normalize_relevance(rel) to convert relevance scores to 0 or 1 (binary)
            * dcg(rels) to compute DCG for a ranked relevance list
        - build a qrels dict: qrels[query_id][doc_id] = normalized relevance
        - skip queries that have no relevance judgments
        - use self.retrieve(query_text, k) to get ranked (doc_id, score) pairs
        - for duplicate qrel entries keep the maximum relevance
    """
        
    if metrics is None:
        metrics = ["recall", "precision", "mrr", "ndcg"]

    def normalize_relevance(rel):
        return 1 if rel > 0 else 0
    
    def dcg(rels):
        return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))
    
    qrels_dict = defaultdict(dict) # qrels[query_id][doc_id] = normalized relevance
    
    for qrel in qrels_iter:
        qid = qrel.query_id
        doc_id = qrel.doc_id
        rel = normalize_relevance(qrel.relevance)
        
        if doc_id in qrels_dict[qid]:
            qrels_dict[qid][doc_id] = max(qrels_dict[qid][doc_id], rel)
        else:
            qrels_dict[qid][doc_id] = rel
            
    results = {metric: [] for metric in metrics}
    
    for query in queries_iter:
        qid = query.query_id
        qtext = query.text
        
        if qid not in qrels_dict:
            continue
        
        relevant_docs = {doc_id: rel for doc_id, rel in qrels_dict[qid].items() if rel > 0}
        num_relevant = len(relevant_docs)
        
        retrieved_docs = self.retrieve(qtext, k)
        
        retrieved_doc_ids = [doc_id for doc_id, _ in retrieved_docs]
        retrieved_rels = [relevant_docs.get(doc_id, 0) for doc_id in retrieved_doc_ids]
        
        if "recall" in metrics:
            recall = sum(retrieved_rels) / num_relevant if num_relevant > 0 else 0.0
            results["recall"].append(recall)
        
        if "precision" in metrics:
            precision = sum(retrieved_rels) / k
            results["precision"].append(precision)
        
        if "mrr" in metrics:
            mrr = 0.0
            for rank, rel in enumerate(retrieved_rels, 1):
                if rel > 0:
                    mrr = 1 / rank
                    break
            results["mrr"].append(mrr)
        
        if "ndcg" in metrics:
            ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
            idcg = dcg(ideal_rels)
            ndcg = dcg(retrieved_rels) / idcg if idcg > 0 else 0.0
            results["ndcg"].append(ndcg)
            
    avg_results = {}
    for metric in metrics:
        values = results[metric]
        avg_results[metric] = sum(values) / len(values) if values else 0.0
        
    return avg_results

BM25.evaluate = evaluate

### Task 3b: Grid Search

Implement and perform grid search for k1 ∈ {0.60, 1.20, 1.80} and b ∈ {0.25, 0.50, 0.75}

In [15]:
def grid_search(
    self,
    queries_iter_factory,
    qrels_iter_factory,
    k: int = 10,
    k1_values: Tuple = (0.60, 1.20, 1.80),
    b_values: Tuple = (0.25, 0.50, 0.75),
) -> List[Dict]:
    """
    Evaluate all combinations of k1 and b, return results as a list of dicts.

    Args:
        queries_iter_factory : callable that returns a fresh queries iterator each time
        qrels_iter_factory   : callable that returns a fresh qrels iterator each time
        k                    : number of top documents to evaluate
        k1_values            : values of k1 to try
        b_values             : values of b to try

    Returns:
        List of dicts, one per (k1, b) combination:
        [
            {"k1": 0.60, "b": 0.25, "recall": ..., "precision": ..., "mrr": ..., "ndcg": ...},
            ...
        ]

    Note:
        - update self.k1 and self.b directly before each self.evaluate() call
        - pass factories/lambdas not iterators as ir_datasets iterators are single-pass,
          so call queries_iter_factory() and qrels_iter_factory() inside the loop to get a fresh iterator each time
    """

    results = []
    
    for k1 in k1_values:
        for b in b_values:
            self.k1 = k1
            self.b = b
            
            eval_results = self.evaluate(queries_iter_factory(), qrels_iter_factory(), k)
            eval_results.update({"k1": k1, "b": b})
            results.append(eval_results)
    
    return results

# attach to BM25 class
BM25.grid_search = grid_search

#### Run the cell below to perform the grid search

In [16]:
# Grid search
# Make sure you have run the Check Task 2 cell first to initialize bm25
print("\nRunning grid search...")
results = bm25.grid_search(
    queries_iter_factory=dataset.queries_iter,
    qrels_iter_factory=dataset.qrels_iter,
    k=10,
)

# Display results table
import pandas as pd

df = pd.DataFrame(results)
df = df.set_index(["k1", "b"])
df.columns = ["Recall@10", "Precision@10", "MRR@10", "nDCG@10"] # change if your metric order is different
print("\n", df.to_string(float_format="%.4f"))





Running grid search...

           Recall@10  Precision@10  MRR@10  nDCG@10
k1  b                                             
0.6 0.25     0.3745        0.2169  0.5173   0.3623
    0.50     0.3845        0.2231  0.5251   0.3734
    0.75     0.3896        0.2280  0.5229   0.3773
1.2 0.25     0.3922        0.2293  0.5307   0.3803
    0.50     0.4017        0.2382  0.5315   0.3899
    0.75     0.3992        0.2396  0.5353   0.3922
1.8 0.25     0.3976        0.2364  0.5360   0.3877
    0.50     0.4005        0.2409  0.5406   0.3930
    0.75     0.4086        0.2458  0.5417   0.3992


#### Report and justify your choice of best configuration



**Which do you consider your best configuration:** k1 = 1.8, b = 0.75

**Why:** Highest scores across all four metrics (Recall .4086, Precision .2458, MRR, .5417, nDCG 0.3992). Consistant improvment with increasing k1 suggetst that the corpus beenfits from less aggressive term-frequency saturation. The hiogher b applies stronger document-length norm, thats good since cranfield docs vary in length.

## Task 4: Error Analysis

### Pick two queries, one where BM25 retrieved great documents, one where BM25 failed the retrieval.

NOTE: I had some help using claude to complete this exercise.


In [17]:
# Find good and bad retrieval examples
bm25.k1 = 1.8
bm25.b = 0.75

# Build qrels dict
from collections import defaultdict
qrels_dict = defaultdict(dict)
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0:
        qrels_dict[qrel.query_id][qrel.doc_id] = qrel.relevance

# Score each query by how many relevant docs appear in top-5
query_scores = []
for query in dataset.queries_iter():
    qid = query.query_id
    if qid not in qrels_dict:
        continue
    top5 = bm25.retrieve(query.text, k=5)
    hits = sum(1 for doc_id, _ in top5 if doc_id in qrels_dict[qid])
    query_scores.append((qid, hits, query.text.strip()))

query_scores.sort(key=lambda x: x[1], reverse=True)
print("=== BEST ===")
for qid, hits, text in query_scores[:3]:
    print(f"QID {qid} | hits={hits} | {text[:80]}")
print("\n=== WORST ===")
for qid, hits, text in query_scores[-3:]:
    print(f"QID {qid} | hits={hits} | {text[:80]}")


=== BEST ===
QID 25 | hits=5 | does a practical flow follow the theoretical concepts for the
interaction betwee
QID 67 | hits=5 | can series expansions be found for the boundary layer on a flat plate in
a shear
QID 3 | hits=4 | what problems of heat conduction in composite slabs have been solved so
far .

=== WORST ===
QID 216 | hits=0 | what investigations have been made of the wave system created by a
static pressu
QID 219 | hits=0 | what are the general effects on flow fields when the reynolds number is
small .
QID 224 | hits=0 | in practice, how close to reality are the assumptions that the flow
in a hyperso


In [19]:
bm25.k1 = 1.8
bm25.b = 0.75

for target_qid in ['25', '224']:
    for query in dataset.queries_iter():
        if query.query_id == target_qid:
            print(f"\n=== QID {target_qid} ===")
            print(f"Query: {query.text.strip()}")
            print(f"Relevant docs: {list(qrels_dict[target_qid].keys())}")
            top5 = bm25.retrieve(query.text, k=5)
            print("\nTop 5 retrieved:")
            for rank, (doc_id, score) in enumerate(top5, 1):
                rel = qrels_dict[target_qid].get(doc_id, 0)
                title = bm25.docs_store[doc_id]['title'].strip().replace('\n', ' ')
                print(f"  {rank}. [{doc_id}] score={score:.3f} rel={rel} | {title[:70]}")



=== QID 25 ===
Query: does a practical flow follow the theoretical concepts for the
interaction between adjacent blade rows of a supersonic cascade .
Relevant docs: ['213', '212', '214', '215', '216', '276', '277', '426', '427']

Top 5 retrieved:
  1. [277] score=25.077 rel=3 | study of flow conditions and deflection angle at exit of two-dimension
  2. [214] score=21.812 rel=3 | on the testing of supersonic compressor cascades .
  3. [215] score=21.010 rel=3 | the test performance of highly loaded turbine stages designed for high
  4. [216] score=18.550 rel=3 | the supersonic axial flow compressor .
  5. [213] score=18.021 rel=2 | the performance of supersonic turbine nozzles .

=== QID 224 ===
Query: in practice, how close to reality are the assumptions that the flow
in a hypersonic shock tube using nitrogen is non-viscous and in
thermodynamic equilibrium .
Relevant docs: ['656', '1313', '1317', '1316', '1318', '1319', '1157', '1274']

Top 5 retrieved:
  1. [1312] score=31.336 rel=0 

#### Query 1 — Successful Retrieval

**Query ID:** 25 

**Query Text:** does a practical flow follow the theoretical concepts for the

**Top 5 Retrieved Documents:** 
  1. [277] score=25.077 rel=3 | study of flow conditions and deflection angle at exit of two-dimension
  2. [214] score=21.812 rel=3 | on the testing of supersonic compressor cascades .
  3. [215] score=21.010 rel=3 | the test performance of highly loaded turbine stages designed for high
  4. [216] score=18.550 rel=3 | the supersonic axial flow compressor .
  5. [213] score=18.021 rel=2 | the performance of supersonic turbine nozzles .


**Does the ranking make sense?** Yes — all 5 retrieved documents are relevant (relevance 2 or 3). The query uses precise technical vocabulary ("supersonic cascade", "blade rows", "flow") that appears directly in the relevant documents. BM25 excels here because there is strong lexical overlap between query terms and document content after stemming, with no vocabulary mismatch.


**Proposed Improvement**  The retrieval is already near-ideal. One enhancement would be to additionally boost documents that also contain the term "theoretical" to better match the comparative ("practical vs. theoretical") framing of the query.

#### Query 2 — Unsuccessful Retrieval

**Query ID:** 224

**Query Text:** in practice, how close to reality are the assumptions that the flow in a hypersonic shock tube using nitrogen is non-viscous and in thermodynamic equilibrium.

**Top 5 Retrieved Documents:**
1. [1312] tabulated solutions of the equilibrium gas properties behind incident normal shocks (rel=0)
2. [317] non-equilibrium flow of an ideal dissociating gas (rel=0)
3. [1286] equilibrium real-gas performance charts for a hypersonic shock-tube (rel=0)
4. [401] inviscid hypersonic airflows with coupled non-equilibrium processes (rel=0)
5. [1296] non-equilibrium expansions of air with coupled chemical reactions (rel=0)

**Does the ranking make sense?** No — all 5 retrieved documents are non-relevant (rel=0), despite having high BM25 scores. The failure is a classic case of **vocabulary mismatch with context confusion**: BM25 latches onto surface keywords like "equilibrium", "hypersonic", and "non-viscous/inviscid" that appear prominently in these documents, but those documents discuss different aspects (e.g. gas property tables, non-equilibrium flows of dissociating gases) rather than validating the specific physical assumptions for nitrogen shock tubes. The actual relevant documents (e.g. 656, 1313–1319) likely discuss the validity of the inviscid and equilibrium assumptions explicitly, but may not repeat those exact terms as frequently.

**Proposed Improvement:** BM25 cannot distinguish documents that *test or validate an assumption* from documents that simply *use* that assumption. A semantic retrieval model (e.g. a dense bi-encoder fine-tuned on scientific text) would better capture the intent of the query. Alternatively, pseudo-relevance feedback — expanding the query with terms from an initial top-k set of retrieved documents — could surface the actually relevant documents in a second-pass retrieval.
